In [ ]:
# Install necessary libraries
!pip install accelerate -U
!pip install sentencepiece
!pip install transformers
!pip install torch
!pip install sacrebleu
!pip install rouge_score

  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinu

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, DataCollatorForSeq2Seq, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
import torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
dataset_path = '/content/drive/MyDrive/ESM_eval_filteredDescription_with_sequences.csv'
df = pd.read_csv(dataset_path)

# Rename columns for consistency
df = df.rename(columns={'sequence': 'text', 'concatenated_description': 'summary'})


In [ ]:
# Split the dataset
train_data, temp_data = train_test_split(df, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)


In [ ]:
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained("t5-small")


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
 def preprocess_function(examples):
    inputs = examples["text"]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(text_target=examples["summary"], max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_encodings = [preprocess_function(item) for item in train_data.to_dict('records')]
val_encodings = [preprocess_function(item) for item in val_data.to_dict('records')]


In [ ]:
def convert_to_numpy(encodings):
    keys = encodings[0].keys()
    data = {key: np.array([np.array(encoding[key]) for encoding in encodings], dtype=np.int64) for key in keys}
    return data

train_encodings = convert_to_numpy(train_encodings)
val_encodings = convert_to_numpy(val_encodings)

In [ ]:
class SummaryDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx], dtype=torch.int64) for key, val in self.encodings.items()}

train_dataset = SummaryDataset(train_encodings)
val_dataset = SummaryDataset(val_encodings)

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model="t5-small")

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=2,
    predict_with_generate=True,
    fp16=True,
    gradient_accumulation_steps=2,
)

# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1494: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
trainer.train()


/usr/local/lib/python3.10/dist-packages/transformers/data/data_collator.py:656: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:274.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Epoch,Training Loss,Validation Loss
1,No log,3.624366
2,No log,3.536324


TrainOutput(global_step=176, training_loss=4.312937649813565, metrics={'train_runtime': 15539.5465, 'train_samples_per_second': 0.362, 'train_steps_per_second': 0.011, 'total_flos': 760620924272640.0, 'train_loss': 4.312937649813565, 'epoch': 2.0})

In [ ]:
trainer.save_model("/content/drive/MyDrive/no_retrieval_")

In [ ]:
from transformers import pipeline

# Check if CUDA is available
device = 0 if torch.cuda.is_available() else -1

# Create the summarization pipeline
summarizer = pipeline("summarization", model=model, tokenizer=tokenizer, device=device)

# BLEU and ROUGE scores
from sacrebleu import corpus_bleu
from rouge_score import rouge_scorer


In [ ]:
def evaluate_model(test_data, summarizer, num_samples=5):
    predictions = []
    references = []

    for item in test_data.to_dict('records'):
        # Truncate input text to the maximum allowed length for the model
        inputs = tokenizer(item['text'], max_length=512, truncation=True, return_tensors="pt").input_ids
        if device >= 0:
            inputs = inputs.to(device)
        input_length = inputs.size(1)

        # Set max_length for summary dynamically based on input length
        max_length = min(128, input_length // 2)

        # Generate summary
        summary = summarizer(item['text'], max_length=max_length, min_length=30, do_sample=False)
        pred = summary[0]['summary_text']
        predictions.append(pred)
        references.append(item['summary'])

    # BLEU score
    from sacrebleu import corpus_bleu
    bleu_score = corpus_bleu(predictions, [references]).score

    # ROUGE score
    from rouge_score import rouge_scorer
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = [scorer.score(pred, ref) for pred, ref in zip(predictions, references)]

    # Average ROUGE scores
    avg_rouge_scores = {
        'rouge1': np.mean([score['rouge1'].fmeasure for score in rouge_scores]),
        'rouge2': np.mean([score['rouge2'].fmeasure for score in rouge_scores]),
        'rougeL': np.mean([score['rougeL'].fmeasure for score in rouge_scores])
    }

    for i in range(min(num_samples, len(test_data))):
        print(f"Original: {test_data.iloc[i]['summary']}")
        print(f"Predicted: {predictions[i]}\n")

    return bleu_score, avg_rouge_scores

In [ ]:
bleu_score, avg_rouge_scores = evaluate_model(test_data, summarizer)

print(f"BLEU Score: {bleu_score}")
print(f"ROUGE Scores: {avg_rouge_scores}")

Your min_length=30 must be inferior than your max_length=28.
/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1273: UserWarning: Unfeasible length constraints: `min_length` (30) is larger than the maximum possible length (28). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length.
  warnings.warn(
Your min_length=30 must be inferior than your max_length=23.
/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1273: UserWarning: Unfeasible length constraints: `min_length` (30) is larger than the maximum possible length (23). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length.
  warnings.warn(
Your min_length=30 must be inferior than your max_length=25.
/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1273: UserWarning: Unfeasible length constraints: `min_length` (30) is large

Original: Hypha-specific G1 cyclin-related protein involved in regulation of morphogenesis and opaque cells filamentous growth, and required for both conventional and pheromone-stimulated biofilm formation. Required to maintain hyphal tip localization of actin and SPA2. Regulates the CDC28 kinase during hyphal growth. The CDC28-HGC1 complex phosphorylates and prevents RGA2 from localizing to hyphal tips, leading to localized CDC42 activation for hyphal extension. The CDC28-HGC1 complex also phosphorylates SEC2 and maintains CDC11 phosphorylation throughout hyphal growth. Moreover CDC28-HGC1 phosphorylation of EFG1 represses cell separation genes during hyphal growth. Also partially controls SEP7 phosphorylation status and subsequent septin ring dynamics. Required for virulence and especially mediates dynamic adhesion to endothelium of blood vessels during circulation. Plasma membrane transporter which mediates resistance to water-soluble, monocarboxylic acids with chain lengths of from